# YAMNet Transfer Training

A scikit-learn head on frozen YAMNet embeddings, run by `src/yamnet_transfer.py`.

This is the notebook for that script, not a second copy of it: every step below
calls the project module, so what runs here is what runs locally.

Two stages. **Extract** turns every dataset segment into a 2048-value YAMNet
embedding and caches it. **Train** fits a sweep of candidates on the training
split, picks the winner on validation macro-F1, and only then scores the test
split once.

Extract needs TensorFlow, which is why this runs on Colab. Train does not, so
once `features.npz` exists you can rerun the sweep on your own machine.

In [ ]:
# Colab already has TensorFlow, so we only pin the audio and sklearn parts.
# scikit-learn matters most: the model is pickled here and unpickled by the app,
# and a different version there can fail to load it.
!pip install -q scikit-learn==1.9.1 librosa==1.0.0 soundfile==0.14.0 joblib==1.6.0 tensorflow_hub==0.16.1

print()
print("If Colab shows a 'Restart runtime' button above, click it, then carry on")
print("from the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

# The embedding cache is thousands of tiny files. On Drive that crawls, so it is
# written to the VM's local disk instead. It is a rebuildable intermediate, and
# features.npz below still lands with the project.
CACHE_PATH = '/content/yamnet_transfer_cache'

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError(f"PROJECT_PATH does not exist: {PROJECT_PATH}")

if not os.path.isfile(os.path.join(PROJECT_PATH, 'src', 'yamnet_transfer.py')):
    raise RuntimeError(f"{PROJECT_PATH} does not look like the project folder.")

# Set before the import: the module reads this when it defines its paths.
os.environ['YAMNET_TRANSFER_CACHE'] = CACHE_PATH

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

try:
    from config.config import CLASSES, PYTHON_MODELS
    from src import yamnet_transfer
except Exception as error:
    raise RuntimeError("Could not import the project modules from PROJECT_PATH.") from error

if not os.path.abspath(yamnet_transfer.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError("yamnet_transfer.py was imported from the wrong place.")

print("imported from   ", yamnet_transfer.__file__)
print("classes         ", len(CLASSES))
print("feature version ", yamnet_transfer.FEATURE_VERSION)
print("embedding length", yamnet_transfer.EMBEDDING_LENGTH)
print("cache           ", yamnet_transfer.CACHE_DIR)
print("outputs         ", yamnet_transfer.RUN_DIR)

In [ ]:
# YAMNet itself. If python_models/yamnet/ came up with the project this loads it
# straight from Drive; otherwise it is downloaded once and kept there, so the
# app works offline later.
from feature_extraction.embeddings import download_yamnet, yamnet_is_downloaded

if yamnet_is_downloaded():
    print("YAMNet is already on Drive")
else:
    download_yamnet()

yamnet_transfer.require_yamnet()
print("YAMNet loaded")

In [ ]:
# The three split folders must all be present and complete. This raises rather
# than training on a broken dataset: it also rejects the same audio appearing
# under two labels, or in two splits at once.
rows = yamnet_transfer.collect_sources()

counts = {}
for row in rows:
    counts[row['dataset_split']] = counts.get(row['dataset_split'], 0) + 1

print(len(rows), "source files")
print("per split:", counts)
print("unique files:", len({row['source_hash'] for row in rows}))

In [ ]:
# Stage one. Roughly one segment per file, a few minutes on CPU; no GPU needed.
# Reruns are cheap because anything already cached is reused.
yamnet_transfer.extract()

In [ ]:
# Stage two. Every candidate is fitted on the training split only, validation
# picks the winner, and the test split is scored once, after that choice.
#
# ACTIVATE=True writes python_models/selection.json, which makes this the model
# the app serves. Leave it False to train without changing what is live.
ACTIVATE = False

test = yamnet_transfer.train(activate=ACTIVATE)

In [ ]:
# Every candidate's validation score, so the choice is visible rather than
# implied by the winner alone.
import json
import os

if not os.path.exists(yamnet_transfer.VALIDATION_FILE):
    raise SystemExit("No validation results yet - run the train cell first.")

with open(yamnet_transfer.VALIDATION_FILE) as handle:
    results = json.load(handle)

print(f"{'candidate':22} {'view':10} {'accuracy':>9} {'macro f1':>9}")
for name, entry in sorted(results.items(), key=lambda kv: -kv[1]['macro_f1']):
    print(f"{name:22} {entry['view']:10} {entry['accuracy']:9.4f} {entry['macro_f1']:9.4f}")

In [ ]:
# What this notebook wrote, and where.
import json
import os

settings = PYTHON_MODELS['yamnet_transfer']

wrote = [
    ('python_models', settings['model_file']),
    ('reports', settings['metrics_file']),
    ('data/yamnet_transfer', 'features.npz'),
    ('data/yamnet_transfer', 'manifest.json'),
    ('data/yamnet_transfer', 'excluded.json'),
    ('data/yamnet_transfer', 'validation_results.json'),
]

for folder, name in wrote:
    path = os.path.join(PROJECT_PATH, folder, name)
    size = f"{os.path.getsize(path) / 1e6:.1f} MB" if os.path.exists(path) else ''
    print('saved' if os.path.exists(path) else 'MISSING', f"{folder}/{name}", size)

# Read back from the metrics file, so this cell stands on its own.
metrics_path = os.path.join(PROJECT_PATH, 'reports', settings['metrics_file'])

if not os.path.exists(metrics_path):
    raise SystemExit("No metrics yet - run the train cell first.")

with open(metrics_path) as handle:
    report = json.load(handle)

print()
print("winner      ", report['candidate'], f"({report['feature_view']})")
print("test accuracy", report['accuracy'], " macro f1", report['macro_f1'])
print()
print("critical-class recall")
for label, value in report['critical_recall'].items():
    flag = '' if value >= 0.85 else '   <- below the 0.85 target'
    print(f"  {label:22} {value}{flag}")

## Bringing it back

Everything above is already on Drive, inside the project folder. Copy these two
into the same folders locally and the app will serve the model:

- `python_models/yamnet_transfer.joblib`
- `reports/yamnet_transfer_metrics.json`

Then either run `python src/yamnet_transfer.py --activate`, or write
`python_models/selection.json` yourself with `{"selected": "yamnet_transfer"}`.
Deleting that file falls back to `ACTIVE_PYTHON_MODEL` in `config/config.py`.

Also worth copying back:

- `data/yamnet_transfer/features.npz` (3,000 x 2,048 float32, so 25 MB before
  compression - the cell above prints the real size)

With that file present, `python src/yamnet_transfer.py` retrains the whole sweep
locally in seconds and needs no TensorFlow, because only extraction does. That is
the loop to use while trying different candidates.

`python_models/yamnet/` is the frozen YAMNet base, about 17 MB. Copy it back too
if you want the app to produce embeddings offline; it is unrelated to the head
trained here.